# 23 — RAG Hyperparameter Tuning
Sweeps key RAG configuration parameters to find the best accuracy/cost tradeoff:
- `top_k` sweep: 3, 5, 10, 15
- Embedding model comparison: `all-MiniLM-L6-v2` vs `all-mpnet-base-v2`
- Context length comparison: 512 vs 1024 tokens per chunk

Best configuration saved to `config/rag.yaml`.
Results saved to `data/results/rag_hyperparams.json`.

In [1]:
import sys, os, json, time
from pathlib import Path
from dotenv import load_dotenv

# Resolve repo root reliably whether run via papermill or jupyter
def _find_repo_root():
    p = Path(os.path.abspath("."))
    for _ in range(6):
        if (p / "prompt-search").exists() and (p / "prompt-search" / "data").exists():
            return p / "prompt-search"
        if (p / "data" / "domain" / "courses.faiss").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not find repo root")
REPO_ROOT = _find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
load_dotenv(REPO_ROOT / ".env")

from retrieval.retriever import CourseRetriever
from retrieval.context_builder import ContextBuilder
from llm.claude_client import ClaudeClient
from evaluation.metrics import EvaluationMetrics

INDEX_DIR   = REPO_ROOT / "data" / "domain"
RESULTS_DIR = REPO_ROOT / "data" / "results"
CONFIG_DIR  = REPO_ROOT / "config"
TEST_CASES  = REPO_ROOT / "data" / "domain" / "test_cases.json"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

client = ClaudeClient(model="claude-haiku-4-5", api_key=os.getenv("ANTHROPIC_API_KEY"))

raw = json.loads(TEST_CASES.read_text())
# Use 40-case subset to limit cost across multiple sweeps
test_cases = (raw["cases"] if isinstance(raw, dict) else raw)[:40]
ground_truth = [tc["expected_answer"] for tc in test_cases]
print(f"Loaded {len(test_cases)} cases for hyperparameter sweep.")

/Users/jaime/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jaime/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 40 cases for hyperparameter sweep.


## Sweep 1: top_k (3, 5, 10, 15)

In [ ]:
def eval_topk(top_k, retriever):
    b = ContextBuilder(retriever, top_k=top_k)
    preds, tokens = [], 0
    t0 = time.time()
    for tc in test_cases:
        prompt, _ = b.build(tc["question"])
        tokens += len(prompt.split())
        resp = client.generate(prompt, temperature=0.0, max_tokens=400)
        preds.append(resp)
    elapsed = time.time() - t0
    acc = EvaluationMetrics.accuracy(preds, ground_truth)
    avg_tokens = tokens / len(test_cases)
    est_cost = (tokens / 1000) * 0.00025
    return {"top_k": top_k, "accuracy": round(acc, 4),
            "avg_prompt_words": round(avg_tokens, 1),
            "est_cost_usd": round(est_cost, 5),
            "elapsed_s": round(elapsed, 1)}

retriever_default = CourseRetriever(index_dir=str(INDEX_DIR))
topk_results = []
print(f"{'top_k':>6} {'accuracy':>10} {'avg_words':>10} {'est_cost':>10} {'time':>8}")
print("-" * 50)
for k in [3, 5, 10, 15]:
    r = eval_topk(k, retriever_default)
    topk_results.append(r)
    print(f"{r['top_k']:>6} {r['accuracy']:>10.2%} {r['avg_prompt_words']:>10.1f} {r['est_cost_usd']:>10.5f} {r['elapsed_s']:>7.1f}s")

 top_k   accuracy  avg_words   est_cost     time
--------------------------------------------------
     3     90.00%      550.1    0.00550    49.7s
     5     90.00%      674.3    0.00674    50.8s
    10     87.50%      954.2    0.00954    54.8s
    15     87.50%     1242.0    0.01242    82.0s


: 

## Sweep 2: Embedding Model Comparison

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

def build_retriever_with_model(model_name):
    """Build a retriever using a different embedding model over the same metadata."""
    import types
    r = CourseRetriever(index_dir=str(INDEX_DIR))
    r.model = SentenceTransformer(model_name)
    # Rebuild FAISS index with new embeddings
    texts = [m["chunk_text"] for m in r.metadata]
    embeddings = r.model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
    embeddings = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9)
    new_index = faiss.IndexFlatIP(embeddings.shape[1])
    new_index.add(embeddings.astype(np.float32))
    r.index = new_index
    return r

embedding_results = []
for model_name in ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]:
    print(f"Building index with {model_name}...")
    t0 = time.time()
    try:
        r = build_retriever_with_model(model_name)
        b = ContextBuilder(r, top_k=5)
        preds = []
        for tc in test_cases:
            prompt, _ = b.build(tc["question"])
            preds.append(client.generate(prompt, temperature=0.0, max_tokens=400))
        acc = EvaluationMetrics.accuracy(preds, ground_truth)
        elapsed = time.time() - t0
        result = {"model": model_name, "accuracy": round(acc, 4), "elapsed_s": round(elapsed, 1)}
        print(f"  {model_name}: {acc:.2%} in {elapsed:.1f}s")
    except Exception as e:
        result = {"model": model_name, "accuracy": None, "error": str(e)}
        print(f"  {model_name}: FAILED — {e}")
    embedding_results.append(result)

Building index with all-MiniLM-L6-v2...
  all-MiniLM-L6-v2: 85.00% in 61.4s
Building index with all-mpnet-base-v2...


## Sweep 3: Context Length (chunk truncation at 512 vs 1024 words)

In [ ]:
def eval_context_length(max_words):
    b = ContextBuilder(retriever_default, top_k=5)
    preds, tokens = [], 0
    t0 = time.time()
    for tc in test_cases:
        prompt, sources = b.build(tc["question"])
        # Truncate context block to max_words
        words = prompt.split()
        if len(words) > max_words:
            prompt = " ".join(words[:max_words])
        tokens += len(words)
        resp = client.generate(prompt, temperature=0.0, max_tokens=400)
        preds.append(resp)
    elapsed = time.time() - t0
    acc = EvaluationMetrics.accuracy(preds, ground_truth)
    est_cost = (tokens / 1000) * 0.00025
    return {"max_context_words": max_words, "accuracy": round(acc, 4),
            "est_cost_usd": round(est_cost, 5), "elapsed_s": round(elapsed, 1)}

context_results = []
print(f"{'max_words':>10} {'accuracy':>10} {'est_cost':>10} {'time':>8}")
print("-" * 44)
for max_words in [512, 1024]:
    r = eval_context_length(max_words)
    context_results.append(r)
    print(f"{r['max_context_words']:>10} {r['accuracy']:>10.2%} {r['est_cost_usd']:>10.5f} {r['elapsed_s']:>7.1f}s")

## Best Configuration & Save

In [ ]:
import yaml

best_topk = max(topk_results, key=lambda r: r["accuracy"])
best_embed = max((r for r in embedding_results if r["accuracy"] is not None), key=lambda r: r["accuracy"])
best_ctx   = max(context_results, key=lambda r: r["accuracy"])

print("Best configuration:")
print(f"  top_k:           {best_topk['top_k']} ({best_topk['accuracy']:.2%})")
print(f"  embedding_model: {best_embed['model']} ({best_embed['accuracy']:.2%})")
print(f"  max_context:     {best_ctx['max_context_words']} words ({best_ctx['accuracy']:.2%})")

rag_config = {
    "top_k": best_topk["top_k"],
    "embedding_model": best_embed["model"],
    "max_context_words": best_ctx["max_context_words"],
    "model": "claude-haiku-4-5",
    "temperature": 0.0,
    "max_tokens": 400,
}

rag_yaml_path = CONFIG_DIR / "rag.yaml"
with open(rag_yaml_path, "w") as f:
    yaml.dump(rag_config, f, default_flow_style=False)
print(f"Config saved to {rag_yaml_path}")

output = {
    "top_k_sweep": topk_results,
    "embedding_model_comparison": embedding_results,
    "context_length_comparison": context_results,
    "best_config": rag_config,
}
out_path = RESULTS_DIR / "rag_hyperparams.json"
out_path.write_text(json.dumps(output, indent=2))
print(f"Results saved to {out_path}")